In [1]:
import findspark
findspark.init()

from pyspark.conf import SparkConf
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

conf = SparkConf().setAppName("626").setMaster("local[4]")
spark = SparkSession.builder.config(conf = conf).getOrCreate()
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/08/26 01:40:11 WARN Utils: Your hostname, de24, resolves to a loopback address: 127.0.1.1; using 192.168.0.103 instead (on interface enp0s3)
25/08/26 01:40:11 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/08/26 01:40:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
'''
Table: Seat

+-------------+---------+
| Column Name | Type    |
+-------------+---------+
| id          | int     |
| student     | varchar |
+-------------+---------+
id is the primary key (unique value) column for this table.
Each row of this table indicates the name and the ID of a student.
id is a continuous increment.
 

Write a solution to swap the seat id of every two consecutive students. 
If the number of students is odd, 
the id of the last student is not swapped.

Return the result table ordered by id in ascending order.

The result format is in the following example.

 

Example 1:

Input: 
Seat table:
+----+---------+
| id | student |
+----+---------+
| 1  | Abbot   |
| 2  | Doris   |
| 3  | Emerson |
| 4  | Green   |
| 5  | Jeames  |
+----+---------+
Output: 
+----+---------+
| id | student |
+----+---------+
| 1  | Doris   |
| 2  | Abbot   |
| 3  | Green   |
| 4  | Emerson |
| 5  | Jeames  |
+----+---------+
Explanation: 
Note that if the number of students is odd, 
there is no need to change the last one's seat.
'''

In [2]:
data = [
(1,'Abbot'  ),
(2,'Doris'  ),
(3,'Emerson'),
(4,'Green'  ),
(5,'Jeames' )
]
schema = ['id','student']

In [3]:
df = spark.createDataFrame(data = data, schema = schema)
df.show()

+---+-------+
| id|student|
+---+-------+
|  1|  Abbot|
|  2|  Doris|
|  3|Emerson|
|  4|  Green|
|  5| Jeames|
+---+-------+



In [5]:
max_id = df.select(F.max(F.col("id"))).collect()[0][0]
df.select(
    F.when( F.col("id")%2 == 0, F.col("id")-1)\
     .when( F.col("id")%2 == 1, F.when(F.col("id") == max_id, F.col("id")).otherwise(F.col("id")+1) ).alias("id"),
    F.col("student")
)\
.orderBy(F.col("id")).show()

+---+-------+
| id|student|
+---+-------+
|  1|  Doris|
|  2|  Abbot|
|  3|  Green|
|  4|Emerson|
|  5| Jeames|
+---+-------+



### SQL Solution

<pre>
SELECT CASE WHEN id%2 = 0 THEN id-1 
            WHEN id%2 = 1 THEN (CASE WHEN id = (SELECT MAX(id) FROM medium_626) THEN id ELSE id+1 END)
            END as id,
       student
FROM medium_626 
order by id
</pre>